In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Installing required packages

In [15]:
!pip install -q openai-whisper openai

# Find the video

In [16]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith((".mp4", ".mov", ".avi", ".mkv", ".webm")):
            print(os.path.join(root, file))

/kaggle/input/datasets/bequeen22/java-video/intellipat java interview questions.mp4


# Extract audio from the video

In [19]:
import subprocess

video_path = "/kaggle/input/datasets/bequeen22/java-video/intellipat java interview questions.mp4"
audio_path = "/kaggle/working/audio.wav"

subprocess.run([
    "ffmpeg",
    "-i", video_path,
    "-vn",
    "-acodec", "pcm_s16le",
    "-ar", "16000",
    "-ac", "1",
    audio_path,
    "-y"
], check=True)

print("Audio extracted successfully!")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Audio extracted successfully!


size=  114060kB time=01:00:49.92 bitrate= 256.0kbits/s speed= 510x    
video:0kB audio:114060kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.000067%


# Convert speech → text using Whisper

In [20]:
import whisper

model = whisper.load_model("base")

result = model.transcribe(
    audio_path,
    fp16=False
)

transcript = result["text"]

print("Transcript:")
print(transcript)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 114MiB/s]


Transcript:
 Welcome to this video and most commonly asked Java interview questions by Intellipart. If you have clicked on this video, there is a high chance that you are already appearing for placement drives. And most often you are being asked interview questions revolving around Java programming language. I mean it's natural because if you want to become a software engineer or a full stack developer, you have to know Java inside out. Even if you are appearing for interviews at MNCs, you will most often find yourself preparing for DSA with either Java or C++. And there is a logic behind that. And that logic is that every enterprise app is being developed by Java itself. Okay, now that we know how important Java is from an interview perspective, let's understand how this video can help you prepare and get the jobs. While preparing for my placement, I focused on Java as my main programming language based on my interview experience and feedback from other Intellipart learners, I have pu

# Import Bart for summarization 

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded on: cpu


# Write the method for summarizing the text 

In [29]:
def summarize_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=200,
            min_length=50,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

# Split the text since our videos can be long 

In [30]:
def split_text(text, max_words=400):
    words = text.split()
    return [
        " ".join(words[i:i + max_words])
        for i in range(0, len(words), max_words)
    ]

chunks = split_text(transcript)

print("Number of chunks:", len(chunks))

Number of chunks: 24


# Summarize by the split chunks of the text

In [31]:
summaries = []

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i + 1}/{len(chunks)}")

    summary = summarize_text(chunk)
    summaries.append(summary)

print("Chunk summarization complete!")

Processing chunk 1/24
Processing chunk 2/24
Processing chunk 3/24
Processing chunk 4/24
Processing chunk 5/24
Processing chunk 6/24
Processing chunk 7/24
Processing chunk 8/24
Processing chunk 9/24
Processing chunk 10/24
Processing chunk 11/24
Processing chunk 12/24
Processing chunk 13/24
Processing chunk 14/24
Processing chunk 15/24
Processing chunk 16/24
Processing chunk 17/24
Processing chunk 18/24
Processing chunk 19/24
Processing chunk 20/24
Processing chunk 21/24
Processing chunk 22/24
Processing chunk 23/24
Processing chunk 24/24
Chunk summarization complete!


# Now join the summary of the chunks, to have our final Summary of the Whole Video

In [33]:
combined_summary = "\n\n".join(
    f"Part {i + 1}:\n{summary}"
    for i, summary in enumerate(summaries)
)

print("\n========== VIDEO SUMMARY ==========\n")
print(combined_summary)


========== VIDEO SUMMARY ==========

Part 1:
Intellipart has put together a list of 30 most commonly asked Java interview questions. This list should help you review important Java topics and get ready for your next interview. While preparing for my placement, I focused on Java as my main programming language.

Part 2:
 Java is known for being simple, secure and able to run on any platform. It's easy to learn thanks to the state forward syntax. Java organizes code through object which makes it easier to work with and maintain. One of the Java's best feature is its ability to run anywhere, thanks to Java virtual machine JDM.

Part 3:
The Java string pool is a huge library where all the string values you create are stored. Whenever you create a Java string in your code, Java checks if that exact string is already in this pool. If it is, it will just give you a reference to the one that's already there.

Part 4:
RAPA classes are asher and sheel for handling primitive data types as object